# Demo Notebook — Trusted AI for Employee Retention

This notebook demonstrates the HR attrition pipeline:
- loading processed data and trained model,
- showing model metrics,
- selecting one employee,
- predicting attrition risk,
- giving a simple explanation,
- displaying fairness results.


In [1]:
import sys
print(sys.executable)

C:\Users\adamt\anaconda3\python.exe


In [2]:
import pandas as pd
import joblib
from pathlib import Path

DATA_PATH = Path('data/processed/hr_model_ready.csv')
MODEL_PATH = Path('models/logistic_regression.joblib')
FAIRNESS_PATH = Path('models/fairness_metrics.csv')
METRICS_PATH = Path('models/model_metrics.csv')

TARGET = 'Termd'
SENSITIVE_COLS = ['Sex', 'RaceDesc', 'HispanicLatino']

df = pd.read_csv(DATA_PATH)
model = joblib.load(MODEL_PATH)
fairness_df = pd.read_csv(FAIRNESS_PATH)
metrics_df = pd.read_csv(METRICS_PATH)

print('Data shape:', df.shape)
print('Loaded model and evaluation files successfully.')


Data shape: (311, 27)
Loaded model and evaluation files successfully.


## 1. Main model metrics

In [3]:
metrics_df

,model,accuracy,precision,recall,f1,roc_auc
0,logistic_regression,0.935897,0.956522,0.846154,0.897959,0.991864
1,random_forest,1.000000,1.000000,1.000000,1.000000,1.000000


## 2. Prepare features for prediction

In [4]:
X = df.drop(columns=[TARGET] + [c for c in SENSITIVE_COLS if c in df.columns], errors='ignore')
y = df[TARGET]

print('Feature shape:', X.shape)
print('Target distribution:')
print(y.value_counts())


Feature shape: (311, 23)
Target distribution:
Termd
0    207
1    104
Name: count, dtype: int64


## 3. Select one employee for the demo

In [5]:
employee_index = 0  # change this value during the demo

employee_features = X.iloc[[employee_index]].copy()
employee_full = df.iloc[[employee_index]].copy()

display(employee_full)


,MarriedID,MaritalStatusID,GenderID,DeptID,PerfScoreID,FromDiversityJobFairID,Salary,Termd,PositionID,Position,...,RecruitmentSource,PerformanceScore,EngagementSurvey,EmpSatisfaction,SpecialProjectsCount,DaysLateLast30,Absences,Age,TenureYears,DaysSinceLastReview
0,0,0,1,5,4,0,62506,0,19,Production Technician I,...,LinkedIn,Exceeds,4.6,5,0,0,1,35.5,7.49,-16


## 4. Predict attrition risk

In [6]:
pred_class = model.predict(employee_features)[0]
pred_proba = model.predict_proba(employee_features)[0, 1]

print(f'Predicted class: {pred_class}')
print(f'Attrition probability: {pred_proba:.4f}')

if pred_proba >= 0.75:
    recommendation = 'High risk: schedule a retention interview and review engagement, workload, and compensation.'
elif pred_proba >= 0.50:
    recommendation = 'Moderate risk: manager follow-up and targeted retention actions recommended.'
else:
    recommendation = 'Low risk: continue regular monitoring.'

print('HR recommendation:', recommendation)


Predicted class: 0
Attrition probability: 0.0401
HR recommendation: Low risk: continue regular monitoring.


## 5. Simple local explanation

For Logistic Regression, we can inspect the contribution of transformed features.
This is a lightweight explanation for the demo.

In [7]:
preprocessor = model.named_steps['preprocessor']
classifier = model.named_steps['classifier']

x_trans = preprocessor.transform(employee_features)
feature_names = preprocessor.get_feature_names_out()
coefs = classifier.coef_[0]

contrib = pd.DataFrame({
    'feature': feature_names,
    'value_times_coef': (x_trans.toarray()[0] if hasattr(x_trans, 'toarray') else x_trans[0]) * coefs
}).sort_values('value_times_coef', ascending=False)

print('Top positive contributions toward attrition risk:')
display(contrib.head(10))

print('Top negative contributions against attrition risk:')
display(contrib.tail(10).sort_values('value_times_coef'))


Top positive contributions toward attrition risk:


,feature,value_times_coef
4,num__PerfScoreID,0.739255
50,cat__State_MA,0.531238
14,num__TenureYears,0.491039
34,cat__Position_Production Technician I,0.292764
2,num__GenderID,0.266903
7,num__PositionID,0.099797
3,num__DeptID,0.087556
8,num__EngagementSurvey,0.037076
5,num__FromDiversityJobFairID,0.029008
62,cat__State_UT,-0.000000


Top negative contributions against attrition risk:


,feature,value_times_coef
15,num__DaysSinceLastReview,-2.531448
88,cat__PerformanceScore_Exceeds,-0.529914
84,cat__RecruitmentSource_LinkedIn,-0.282404
12,num__Absences,-0.241236
11,num__DaysLateLast30,-0.239029
13,num__Age,-0.212526
10,num__SpecialProjectsCount,-0.154296
0,num__MarriedID,-0.136312
76,cat__Department_Production,-0.088774
1,num__MaritalStatusID,-0.082337


## 6. Fairness audit results

In [8]:
fairness_df

,group_column,group_value,count,accuracy,recall,false_positive_rate,false_negative_rate
0,Sex,F,43,0.9302,0.8667,0.0357,0.1333
1,Sex,M,35,0.9429,0.8182,0.0000,0.1818
2,RaceDesc,American Indian or Alaska Native,1,1.0000,0.0000,0.0000,0.0000
3,RaceDesc,Asian,10,0.9000,0.6667,0.0000,0.3333
4,RaceDesc,Black or African American,22,1.0000,1.0000,0.0000,0.0000
5,RaceDesc,Two or more races,4,0.7500,0.0000,0.0000,1.0000
6,RaceDesc,White,41,0.9268,0.8667,0.0385,0.1333
7,HispanicLatino,No,73,0.9315,0.8400,0.0208,0.1600
8,HispanicLatino,Yes,5,1.0000,1.0000,0.0000,0.0000


## 7. Demo speaking notes

- We use Logistic Regression as the main model because it is both strong and interpretable.
- The model predicts whether an employee is at risk of leaving.
- We can explain a single prediction by showing the strongest contributing factors.
- We also audit fairness across sensitive groups such as sex and race.
- This is a decision-support tool, not an automated HR decision-maker.
